In [2]:

# Standard Libraries

import os
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# Data Manipulation

import numpy as np
import pandas as pd


# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")


# Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Miscellaneous

from tqdm import tqdm

# Model-Ready Dataset Construction

## Workflow
```mermaid
flowchart LR
    A[Feature Engineered Data] --> B[Tree Models]
    A --> C[Scaling]

    C --> D[Logistic Regression]
    C --> E[Neural Network]
```


In [28]:
DATA_DIR =  Path("data")
CONFIG_DIR = Path("artifacts")
X_train = pd.read_csv(DATA_DIR/'FE_datasets/X_train.csv')
X_valid = pd.read_csv(DATA_DIR/'FE_datasets/X_valid.csv')
X_test = pd.read_csv(DATA_DIR/'FE_datasets/X_test.csv')

y_train = pd.read_csv(DATA_DIR/'FE_datasets/y_train.csv')
y_valid = pd.read_csv(DATA_DIR/'FE_datasets/y_valid.csv')
y_test = pd.read_csv(DATA_DIR/'FE_datasets/y_test.csv')



# 6.1 Encode Remaining Categorical Features

At this point we still have a few low-cardinality categorical columns.

For simplicity and consistency across models, we'll use **Ordinal Encoding**.

In [8]:
# Ordinal Encoding

from sklearn.preprocessing import OrdinalEncoder

In [9]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [11]:
numeric_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(include="object")
    .columns
    .tolist()
)

print("Numeric Features :", len(numeric_features))

print("Categorical Features :", len(categorical_features))

Numeric Features : 729
Categorical Features : 23


Fit **only on training data**.

In [12]:
encoder.fit(
    X_train[categorical_features]
)

,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'use_encoded_value'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",-1
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.",'auto'
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequent categories. If there are infrequentcategories, `max_categories` includes the category representing theinfrequent categories along with the frequent categories. If `None`,there is no limit to the number of output features.`max_categories` do **not** take into account missing or unknowncategories. Setting `unknown_value` or `encoded_missing_value` to aninteger will increase the number of unique integer codes by one each.This can result in up to `max_categories + 2` integer codes... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
Name,Type,Value
categories_ categories_: list of arraysThe categories of each feature determined during ``fit`` (in order ofthe features in X and corresponding with the output of ``transform``).This does not include categories that weren't seen during ``fit``.,list,"[array(['C', '... dtype=object), array(['Missi... dtype=object), array(['Missi... dtype=object), array(['F', '... dtype=object), ...]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](23,)","['ProductCD','card4','card6',...,'id_37','id_38','DeviceType']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 1.0,int,23


Transform all splits.

In [13]:
X_train[categorical_features] = encoder.transform(
    X_train[categorical_features]
)

X_valid[categorical_features] = encoder.transform(
    X_valid[categorical_features]
)

X_test[categorical_features] = encoder.transform(
    X_test[categorical_features]
)

# 6.2 Verify Missing Values

There should be no missing values remaining.

In [14]:
print("Training Missing :", X_train.isnull().sum().sum())

print("Validation Missing :", X_valid.isnull().sum().sum())

print("Testing Missing :", X_test.isnull().sum().sum())

Training Missing : 0
Validation Missing : 390
Testing Missing : 8836


# 6.3 Create Tree-Based Dataset

Tree algorithms don't require scaling.

In [15]:
X_train_tree = X_train.copy()

X_valid_tree = X_valid.copy()

X_test_tree = X_test.copy()

---

# 6.4 Scale Numerical Features

Neural Networks and Logistic Regression converge faster with standardized numerical inputs.

In [16]:
from sklearn.preprocessing import StandardScaler

In [17]:
scaler = StandardScaler()

scaler.fit(
    X_train
)

StandardScaler()

Only the training set is used to estimate the mean and standard deviation, preventing information leakage into validation and test data.



Transform datasets.

In [18]:
X_train["card1_amt_std"] = X_train["card1_amt_std"].fillna(0)
X_test["card1_amt_std"] = X_test["card1_amt_std"].fillna(0)
X_valid["card1_amt_std"] = X_valid["card1_amt_std"].fillna(0)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_valid_scaled = scaler.transform(X_valid)


---

# 6.5 Convert Back to DataFrame

This makes feature importance analysis easier later.

In [19]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_valid_scaled = pd.DataFrame(
    X_valid_scaled,
    columns=X_valid.columns,
    index=X_valid.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

## Impute scaled training set

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

imputer.fit(X_train_scaled)

X_train_scaled = pd.DataFrame(
    imputer.fit_transform(X_train_scaled),
    columns=X_train_scaled.columns,
    index=X_train_scaled.index,
)

X_valid_scaled = pd.DataFrame(
    imputer.transform(X_valid_scaled),
    columns=X_valid_scaled.columns,
    index=X_valid_scaled.index,
)

X_test_scaled = pd.DataFrame(
    imputer.transform(X_test_scaled),
    columns=X_test_scaled.columns,
    index=X_test_scaled.index,
)

---

# 6.6 Feature Selection

Let's remove highly correlated variables.

In [20]:
corr = (
    X_train_tree
    .corr()
    .abs()
)

In [21]:
upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

In [22]:
correlated_features = [
    col
    for col in upper.columns
    if any(
        upper[col] > 0.98
    )
]

In [40]:
# # Tasks too much time! Run on GPU
# # Chack availble HW with linux
# try:
#     import cudf
#     import rmm

#     print("GPU is available.")
# except ImportError:
#     print("RAPIDS/cuDF is not installed.")

In [41]:
# from utils import find_correlated_features
# correlated_features = find_correlated_features(X_train_tree)

In [23]:
print(f"Highly Correlated Features : {len(correlated_features)}")

Highly Correlated Features : 316


---

Remove them.

In [24]:
X_train_tree.drop(
    columns=correlated_features,
    inplace=True
)

X_valid_tree.drop(
    columns=correlated_features,
    inplace=True
)

X_test_tree.drop(
    columns=correlated_features,
    inplace=True
)

Do the same for scaled datasets.

In [25]:
X_train_scaled.drop(
    columns=correlated_features,
    inplace=True
)

X_valid_scaled.drop(
    columns=correlated_features,
    inplace=True
)

X_test_scaled.drop(
    columns=correlated_features,
    inplace=True
)

Removing highly correlated variables can reduce redundancy and training time, particularly for linear models, while tree-based models are generally more tolerant of correlated predictors.

---

# 6.7 Check Class Imbalance

In [29]:
fraud_distribution = (
    y_train
    .value_counts()
)

display(fraud_distribution)

isFraud
0          398840
1           14538
Name: count, dtype: int64

In [30]:
fraud_ratio = (
    y_train
    .value_counts(normalize=True)
)

display(fraud_ratio)

isFraud
0          0.964831
1          0.035169
Name: proportion, dtype: float64

---

# 6.8 Compute Class Weights

Instead of immediately applying SMOTE, we'll first use class weights.

In [31]:
from sklearn.utils.class_weight import compute_class_weight

In [32]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

In [33]:
class_weights = dict(
    zip(classes, weights)
)

print(class_weights)

{np.int64(0): np.float64(0.5182253535252231), np.int64(1): np.float64(14.217155041959003)}


Using class weights is a strong baseline for highly imbalanced fraud detection because it increases the cost of misclassifying minority-class examples without generating synthetic transactions. Many recent IEEE-CIS studies evaluate class weighting before introducing oversampling methods such as SMOTE.

# 6.9 Final Dataset Summary

In [34]:
print("=" * 60)
print("FINAL DATASETS")
print("=" * 60)

print("Tree Training :", X_train_tree.shape)

print("Tree Validation :", X_valid_tree.shape)

print("Tree Testing :", X_test_tree.shape)

print()

print("Scaled Training :", X_train_scaled.shape)

print("Scaled Validation :", X_valid_scaled.shape)

print("Scaled Testing :", X_test_scaled.shape)

FINAL DATASETS
Tree Training : (413378, 436)
Tree Validation : (88581, 436)
Tree Testing : (88581, 436)

Scaled Training : (413378, 436)
Scaled Validation : (88581, 436)
Scaled Testing : (88581, 436)


---

# 6.10 Save Preprocessed Data (Optional)

This avoids repeating preprocessing every time you rerun the notebook.

# 6.11 Save Preprocessed Data

In [35]:
import os

# Define the directory to save the datasets
SAVE_DIR = Path("data/Training_Datasets")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save tree-based datasets
X_train_tree.to_csv(SAVE_DIR / "X_train_tree.csv", index=False)
X_valid_tree.to_csv(SAVE_DIR / "X_valid_tree.csv", index=False)
X_test_tree.to_csv(SAVE_DIR / "X_test_tree.csv", index=False)

# Save scaled datasets
X_train_scaled.to_csv(SAVE_DIR / "X_train_scaled.csv", index=False)
X_valid_scaled.to_csv(SAVE_DIR / "X_valid_scaled.csv", index=False)
X_test_scaled.to_csv(SAVE_DIR / "X_test_scaled.csv", index=False)

# Save target variables
y_train.to_csv(SAVE_DIR / "y_train.csv", index=False)
y_valid.to_csv(SAVE_DIR / "y_valid.csv", index=False)
y_test.to_csv(SAVE_DIR / "y_test.csv", index=False)

print(f"All preprocessed datasets saved to {SAVE_DIR}")

All preprocessed datasets saved to data\Training_Datasets


In [ ]:
artifacts = {

    "imputer": imputer,

    "encoder": encoder,

    "scaler": scaler,

    "feature_columns": list(X_train_tree.columns),

    "threshold": FINAL_THRESHOLD

}

In [ ]:
joblib.dump(
    artifacts,
    CONGIG_DIR / "preprocessing.joblib"
)

print("✓ Preprocessing artifacts saved.")

In [ ]:
# Some memory cleaning stuffs

In [45]:
import gc

def reset_memory():
    # Delete all variables.
    for var in gc.get_objects():
        try:
            del var
        except:
            pass

    # Run the garbage collector.
    gc.collect()

    print("Memory has been reset")


In [ ]:
# del X_train
# del X_valid
# del X_test
# del y_train
# del y_valid
# del y_test
# del X_train_scaled
# del X_test_scaled
# del X_valid_scaled
# del X_train_tree
# del X_test_tree
# del X_valid_tree

In [47]:
reset_memory()

Memory has been reset
